<img src="../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

<img src="assets/agent05-top-light.svg" alt="An outside model, AceFF 2.0, gets a wrapper and plugs into Toolkit beside AIMNet2; a batch passes through it and energy and forces come out." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 5px solid #76b900; border-radius: 4px; padding: 24px 28px; margin: 8px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="font-variant: small-caps; font-size: 12px; font-weight: 700; letter-spacing: 2px; opacity: 0.75;">AGENTIC TRACK, NOTEBOOK 05</div><h1 style="margin: 6px 0 8px; padding: 0; border: 0;"><span style="font-size: 32px; font-weight: 700; line-height: 1.2; color: inherit;">Guest model</span></h1><div style="font-size: 18px; line-height: 1.45; opacity: 0.85;">Learn how you can wrap your own model, or one you just read about, to run in Toolkit.</div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 12px 18px; margin: 12px 0 18px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; line-height: 1.6;"><b>Hardware:</b> a CUDA-capable NVIDIA GPU<br><b>Prerequisites:</b> Meet your agent; Ready-made; Made to order<br><b>References:</b> <a href="https://nvidia.github.io/nvalchemi-toolkit/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit documentation</a>; <a href="https://nvidia.github.io/nvalchemi-toolkit/examples/" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Toolkit examples</a>; <a href="https://github.com/NVIDIA/skills/tree/main/skills/nvalchemi-toolkit" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">the nvalchemi-toolkit skill</a></div>

You will meet this class along the way:

- `BaseModelMixin`: the interface your own model takes on

<img src="assets/agent05-roadmap-light.svg" alt="The steps of this notebook: wrap a new model, set it beside the composed model, relax a batch with it." style="display:block;box-sizing:border-box;width:100%;max-width:920px;height:auto;margin:0;">

In [ ]:
# Load the course helpers and print what your agent needs to start
from helpers import agent05 as helpers
helpers.configure_tutorial()             # course display defaults
work_dir = helpers.make_work_dir()       # notebooks/agent05_work
# The molecule pairs of Made to order, and the composed model's energies on them
scan_batch, scan_labels, composed_energy = helpers.load_pair_scans()
helpers.show_session()                   # prints the session lines

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">1. Wrap a new model</h2>

AceFF 2.0 is an open model for drug-like molecules from outside Toolkit's
shipped models. Wrapping it shows how easily Toolkit takes in an outside
model, so you stay flexible for your research needs.

#### Example: wrap AceFF 2.0

```text
use the nvalchemi-toolkit skill
- wrap a model toolkit has no wrapper for, so i can use it wherever a built-in model is: relaxation, md, pipelines
- model: AceFF 2.0 from Acellera (Acellera/AceFF-2.0 on hugging face), loaded with the torchmd-net package
- it has open weights, covers small organic molecules and takes the total charge
- first tell me what has to be converted between the two libraries
- write agent05_work/aceff_wrapper.py with one wrapper class
- give me aceff_model: built from that class, on the gpu, energy in eV, forces in eV/Å
```

In [ ]:
%load agent05_work/aceff_wrapper.py

In [ ]:
# Show what the wrapped model returns
helpers.check_wrapped_model(aceff_model)

The table lists what the wrapped model returns.

> </> API
>
> `ModelConfig(outputs=, autograd_outputs=, autograd_inputs=, required_inputs=, supports_pbc=, neighbor_config=)`
>
> **Input:** what the model returns and what it needs<br>
> **Effect:** declares both to Toolkit

> </> API
>
> `adapt_output(model_output, data)`
>
> **Input:** the model's output as a dict, and the batch<br>
> **Effect:** returns Toolkit's output dict, with the energy shaped to `[B, 1]`

> ⚙️ **Fine control**
>
> ```python
> class AceFFWrapper(nn.Module, BaseModelMixin):   # nn.Module first
>     # model_config: outputs, autograd inputs, neighbor list
>     # embedding_shapes, compute_embeddings: required, may be stubs
>     def adapt_input(self, data, **kw): ...        # Batch -> model inputs
>     def forward(self, data, **kw): ...            # energy, then forces
> ```

OrbMol v2 is another open model to try next.

#### Try it: the guest beside the composed model

Ask your agent to evaluate the molecule pairs of "Made to order" with the
model you wrapped. The composed model's energies on the same pairs are
already loaded, so the plot sets the two side by side.

| | |
| --- | --- |
| input  | `scan_batch`; `aceff_model` |
| output | `aceff_energy`, in `agent05_work/aceff_pairs.py` |

Done when: each panel shows a curve for the wrapped model beside the
composed model.

In [ ]:
%load agent05_work/aceff_pairs.py

In [ ]:
# Set the wrapped model beside the composed model on the molecule pairs
helpers.plot_interaction_curves(scan_labels, composed=composed_energy, wrapped=aceff_energy)

#### Try it: relax a batch with the guest model

A wrapped model relaxes molecules as a built-in one does. Ask your agent to
relax a batch of molecules with the model you wrapped, the way you relaxed
crystals in "Ready-made".

| | |
| --- | --- |
| input  | molecules of your own, or `helpers.load_molecules()`; `aceff_model` |
| output | `relaxed_batch`, `relaxed_max_force`, `force_threshold`, in `agent05_work/aceff_relax.py` |

Done when: every molecule's largest force is below the threshold.

> 💡 **Hint**
>
> Nothing at hand? `helpers.load_molecules()` gives you a tested default.

In [ ]:
%load agent05_work/aceff_relax.py

In [ ]:
# Show each molecule's largest force beside the threshold
helpers.show_relaxed_forces(relaxed_batch, relaxed_max_force, force_threshold)

<h2 style="display: block; color: inherit; background: transparent; border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; padding: 15px 20px; margin: 24px 0 14px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 20px; font-weight: 500;">Explore more</h2>

Other ways to bring a model into Toolkit.

<div style="display: flex; flex-wrap: wrap; gap: 16px; margin: 4px 0 18px; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif;"><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/advanced/09_uma_nve.py">UMA, a wrapper Toolkit ships</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">Run UMA through Toolkit's integrators, in its own environment.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://github.com/NVIDIA/nvalchemi-toolkit/blob/v0.3.0/examples/distributed/04_byo_pytorch_mpnn.py">Wrap your own PyTorch model</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">A small PyTorch potential wrapped, run and validated across ranks.</div></div><div style="flex: 1 1 190px; min-width: 190px; box-sizing: border-box; background: transparent; color: inherit; border: 1px solid rgba(127,127,127,0.35); border-radius: 4px; padding: 16px 16px 18px; line-height: 1.5; overflow-wrap: break-word;"><img src="assets/explore-workflows-light.svg" alt="" width="24" height="24" style="display: block; width: 24px; height: 24px; margin: 0 0 10px;"><div><b><a href="https://nvidia.github.io/nvalchemi-toolkit/userguide/models.html">Models guide</a></b></div><div style="margin: 8px 0 0; font-size: 14px; color: inherit; opacity: 0.8;">The wrapper interface in full, with a worked example.</div></div></div>

<div style="border: 1px solid rgba(127,127,127,0.35); border-left: 4px solid #76b900; border-radius: 4px; background: transparent; color: inherit; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; padding: 15px 20px; margin: 24px 0 18px; line-height: 1.6;"><h2 style="margin: 0 0 10px; padding: 0; border: 0; font-family: NVIDIA Sans, Helvetica Neue, Arial, sans-serif; font-size: 24px; font-weight: 500; line-height: 1.25; color: inherit;">Notebook 05 recap</h2><div style="margin: 0 0 10px;">Over these three notebooks you relaxed and simulated crystals with a built-in model, composed a model from interaction terms, and wrapped a new model. All three share one interface, so the calls that relax and simulate did not change from one model to the next.</div><div style="margin: 0 0 10px;">Composition and wrapping allow full flexibility to bring the right ingredients and tools for your research: you are never locked by prebuilt defaults. With hooks you can gain even finer control over your simulations.</div><div style="margin: 0;"><img src="assets/fork-agent-light.svg" alt="" width="18" height="18" style="width: 18px; height: 18px; vertical-align: -3px; margin-right: 6px;"><b>Next:</b> <a href="alchemi-agent-06-hooked-in-at-every-step.ipynb" style="color: inherit; text-decoration: underline; text-decoration-color: #76b900; text-decoration-thickness: 2px; text-underline-offset: 3px;">Hooked in at every step →</a></div></div>